# Modern Web Automation with Python and Selenium — Study Notes

These are **working notes while going through the Real Python tutorial**, rather than a copy of the article. I’m keeping the tutorial's major headings, the important ideas, and representative/recreated code so I can actually study from the notebook later.

**Tutorial:** https://realpython.com/modern-web-automation-with-python-and-selenium/

**Main project:** build a text-based Bandcamp Discover music player controlled through Selenium.

> My mental model: Selenium is basically Python driving a real browser. The browser renders JavaScript-heavy pages, and Selenium lets me inspect and interact with the resulting DOM.

The tutorial's big progression is:

1. basic browser setup
2. navigation + locating elements
3. interaction
4. waiting for dynamic content
5. Page Object Model (POM)
6. assemble the music-player application

The POM part is the important architectural payoff: keep selectors and browser-specific interaction details away from the higher-level application logic.


## 1. Understand the Project and Approach

### Why Selenium?

Requests/Beautiful Soup/Scrapy are excellent when the information I need is already available in the HTTP response. Selenium is useful when I need to behave more like a user in a browser, especially when JavaScript creates or updates the content.

For this project, Selenium will:

- launch Firefox
- open Bandcamp Discover
- find track cards
- click Play/Pause
- click "View more"
- collect album/artist/genre information
- run headlessly once the code works

### Page Object Model

The POM idea is to put the knowledge of *how the website works* into page/component classes.

So instead of higher-level code doing:

```text
find CSS selector → wait → click → parse text
```

it can eventually do something more readable like:

```text
track.play()
track.pause()
player.tracklist.load_more()
```

The ugly browser details still exist. They are just confined to the layer where they belong. Humanity has invented architecture largely so that future-us doesn't have to debug past-us's selectors at 2 AM.


## 2. Set Up the Environment

The tutorial uses Python 3.10+ and Firefox + GeckoDriver.

### Virtual environment



In [ ]:
# macOS / Linux
python -m venv venv
source venv/bin/activate

# Windows PowerShell:
# python -m venv venv
# .\\venv\\Scripts\\activate

python --version
python -m pip install selenium

# If following the tutorial's exact pinned environment instead:
# python -m pip install -r requirements.txt


### GeckoDriver

Firefox needs GeckoDriver so Selenium can communicate with the browser.

The tutorial recommends putting `geckodriver` somewhere on `PATH`, then checking it with:



In [ ]:
geckodriver --version


### First sanity check

Before touching Bandcamp, make sure Selenium can actually launch Firefox and close it again.

The tutorial also shows Chrome as an alternative. The only real change is the browser/options import and WebDriver constructor.


In [ ]:
from selenium import webdriver
from selenium.webdriver.firefox.options import Options

options = Options()
options.add_argument("--headless")

driver = webdriver.Firefox(options=options)
driver.get("https://www.python.org")

print(driver.title)

driver.quit()

# Chrome alternative:
# from selenium.webdriver.chrome.options import Options
# options = Options()
# options.add_argument("--headless")
# driver = webdriver.Chrome(options=options)


**Important note:** `.quit()` matters. A headless browser is still a browser process. If I forget to close it, I can accumulate invisible browser instances eating RAM like tiny digital raccoons.


## 3. Navigate a Web Page

### First: inspect the page manually

Before writing selectors, inspect the website's DOM with browser developer tools.

The tutorial's simplified Bandcamp structure is roughly:

- a `results-grid` container
- `results-grid-item` elements for individual tracks
- a `play-pause-button`
- album/artist information inside `.meta`
- optional genre information

The exact HTML can change, so these selectors are observations about the current page, not eternal truths handed down from Selenium mountain.


In [ ]:
from selenium import webdriver
from selenium.webdriver.firefox.options import Options

options = Options()
options.add_argument("--headless")

driver = webdriver.Firefox(options=options)
driver.implicitly_wait(5)

driver.get("https://bandcamp.com/discover/")
print(driver.title)

driver.quit()


### Locating elements

Selenium provides several locator strategies. The tutorial emphasizes `By` + `find_element()` / `find_elements()`.

- `find_element()` → one element
- `find_elements()` → list of elements

IDs and stable classes are generally preferable to gigantic absolute XPath expressions.


In [ ]:
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By

options = Options()
options.add_argument("--headless")

driver = webdriver.Firefox(options=options)
driver.implicitly_wait(5)
driver.get("https://bandcamp.com/discover/")

pagination = driver.find_element(By.ID, "view-more")
print(pagination.accessible_name)

tracks = driver.find_elements(By.CLASS_NAME, "results-grid-item")
print(len(tracks))
print(tracks[0].text)

# Search inside a WebElement, not the entire page:
first_track = tracks[0]
album = first_track.find_element(
    By.CSS_SELECTOR,
    "div.meta a strong",
)
print(album.text)

driver.quit()


### Locator notes

```python
By.ID
By.CLASS_NAME
By.CSS_SELECTOR
By.XPATH
By.TAG_NAME
By.NAME
```

An absolute XPath such as:

```python
# /html/body/div[2]/span[1]/a[3]
```

can work, but it is extremely fragile. If someone inserts one extra `<div>`, congratulations, the automation has become archaeology.

Prefer stable IDs or semantic/stable classes when possible.


## 4. Interact With Web Elements

### Click

The basic operation is simply:



In [ ]:
button = driver.find_element(By.ID, "submit-button")
button.click()


For the Bandcamp project, clicking the "View more" button exposes additional tracks.

The tutorial initially uses `time.sleep()` after the click as a simple demonstration. This is deliberately presented as a stepping stone, not the final synchronization strategy.


In [ ]:
import time

pagination = driver.find_element(By.ID, "view-more")
pagination.click()

time.sleep(0.5)

tracks = driver.find_elements(By.CLASS_NAME, "results-grid-item")
print(len(tracks))

# Better approach appears later: explicit waits.


### Send text / keystrokes

`send_keys()` types into an input. `.submit()` submits its containing form.

There are two equivalent-ish approaches shown:


In [ ]:
search_box = driver.find_element(By.TAG_NAME, "input")
search_box.send_keys("selenium")
search_box.submit()

# Or explicitly send Enter:
# from selenium.webdriver.common.keys import Keys
# search_box.send_keys(Keys.ENTER)


### Cookie/overlay handling

A cookie banner can block interactions even though the target element exists in the DOM.

The tutorial shows a direct click and then the more robust EAFP-style `try/except` version:


In [ ]:
from selenium.common.exceptions import NoSuchElementException

try:
    cookie_button = driver.find_element(
        By.CSS_SELECTOR,
        "#cookie-control-dialog button.g-button.outline",
    )
    cookie_button.click()
except NoSuchElementException:
    pass

# Alternative for some situations:
# driver.execute_script(
#     "arguments[0].click();",
#     overlay_element,
# )
#
# The JavaScript approach can bypass normal interaction behavior,
# so it isn't necessarily appropriate for realistic testing.


### More complex gestures

`ActionChains` can compose interactions such as hovering and clicking.



In [ ]:
from selenium.webdriver import ActionChains

menu = driver.find_element(By.CSS_SELECTOR, ".menu")
submenu = driver.find_element(By.CSS_SELECTOR, ".menu #submenu")

actions = ActionChains(driver)
actions.move_to_element(menu)
actions.click(submenu)
actions.perform()


### Forms

A form can be located first, then its individual inputs can be filled before submitting the form.



In [ ]:
signup = driver.find_element(By.ID, "signup-form")

email = signup.find_element(By.NAME, "email")
password = signup.find_element(By.NAME, "password")

email.send_keys("user@example.com")
password.send_keys("example-password")

signup.submit()


## 5. Handle Dynamic Content

This is one of the most important sections.

### Why `sleep()` is not enough

A fixed sleep says:

> "I hope the website finishes in 0.5 seconds."

An explicit wait says:

> "Continue when this particular condition becomes true, up to 10 seconds."

The second approach is much more robust.

### Implicit wait

An implicit wait applies to element searches throughout a driver session:


In [ ]:
driver.implicitly_wait(5)


### Explicit wait

Use `WebDriverWait` with an expected condition.



In [ ]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 10)

# Wait without using the returned element:
wait.until(
    EC.element_to_be_clickable((By.ID, "view-more"))
)

# Or use the returned element directly:
pagination = wait.until(
    EC.element_to_be_clickable((By.ID, "view-more"))
)
pagination.click()


### Common expected conditions

Useful ones to remember:

- `presence_of_element_located()` → exists in the DOM
- `visibility_of_element_located()` → exists and is visible
- `element_to_be_clickable()` → visible + enabled
- `alert_is_present()`
- `title_is()` / `title_contains()`
- `url_to_be()` / `url_contains()`

The key idea is to wait for a *meaningful state*, not an arbitrary amount of time.


### Custom wait conditions

I can also pass my own function to `wait.until()`. This is useful when none of Selenium's built-in conditions express the exact state I care about.


In [ ]:
wait = WebDriverWait(driver, 10)

def tracks_loaded(driver):
    cards = driver.find_elements(
        By.CLASS_NAME,
        "results-grid-item",
    )
    return any(card.text.strip() for card in cards)

wait.until(tracks_loaded)


### Synchronization notes

Even explicit waits can be flaky if I choose the wrong signal.

I want to wait for the most stable indication that the page has finished its asynchronous work. Headless mode can also behave differently from visible mode, so removing `--headless` is a useful debugging technique.

JavaScript alerts can be handled similarly:


In [ ]:
from selenium.common.exceptions import NoAlertPresentException

try:
    alert = driver.switch_to.alert
    alert.dismiss()

    # Or:
    # alert.accept()
except NoAlertPresentException:
    pass


## 6. Implement the Page Object Model (POM)

### Project structure

The tutorial separates the browser-facing code into:

```text
bandcamp/
├── __init__.py
├── base.py
├── elements.py
├── locators.py
└── pages.py
```

Conceptually:

```text
DiscoverPage
    └── TrackListElement
            └── TrackElement
```

`locators.py` owns selectors, `elements.py` owns component behavior, and `pages.py` describes the page-level object.

This means the application code can eventually stop caring what CSS selector Bandcamp happens to use today.


### `base.py`

The base classes centralize common WebDriver setup, window sizing, and waiting behavior.


In [ ]:
from selenium.webdriver.remote.webdriver import WebDriver
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.support.wait import WebDriverWait

MAX_WAIT_SECONDS = 10.0
DEFAULT_WINDOW_SIZE = (1920, 3000)


class WebPage:
    def __init__(self, driver: WebDriver) -> None:
        self._driver = driver
        self._driver.set_window_size(*DEFAULT_WINDOW_SIZE)
        self._driver.implicitly_wait(5)
        self._wait = WebDriverWait(driver, MAX_WAIT_SECONDS)


class WebComponent(WebPage):
    def __init__(self, parent: WebElement, driver: WebDriver) -> None:
        super().__init__(driver)
        self._parent = parent


### `pages.py`

The page object represents only the parts of the Discover page that the application needs.

It handles cookie consent and exposes the track list.


In [ ]:
from selenium.common.exceptions import NoSuchElementException

from bandcamp.base import WebPage
from bandcamp.elements import TrackListElement
from bandcamp.locators import DiscoverPageLocator


class DiscoverPage(WebPage):
    def __init__(self, driver):
        super().__init__(driver)
        self._accept_cookie_consent()

        results = self._driver.find_element(
            *DiscoverPageLocator.DISCOVER_RESULTS
        )
        self.discover_tracklist = TrackListElement(
            results,
            self._driver,
        )

    def _accept_cookie_consent(self):
        try:
            self._driver.find_element(
                *DiscoverPageLocator.COOKIE_ACCEPT_NECESSARY
            ).click()
        except NoSuchElementException:
            pass


### `elements.py`: track list

The track list component waits for track text, finds all track cards, filters out invisible/empty ones, and wraps them in `TrackElement` objects.


In [ ]:
from selenium.webdriver.support import expected_conditions as EC

from bandcamp.base import WebComponent
from bandcamp.locators import TrackListLocator


class TrackListElement(WebComponent):
    def __init__(self, parent, driver=None):
        super().__init__(parent, driver)
        self.available_tracks = self._get_available_tracks()

    def load_more(self):
        button = self._driver.find_element(
            *TrackListLocator.PAGINATION_BUTTON
        )
        button.click()

        self._wait.until(
            EC.element_to_be_clickable(
                TrackListLocator.PAGINATION_BUTTON
            )
        )
        self.available_tracks = self._get_available_tracks()

    def _get_available_tracks(self):
        self._wait.until(
            self._track_text_loaded,
            message="Timeout waiting for track text to load",
        )

        cards = self._driver.find_elements(*TrackListLocator.ITEM)

        return [
            TrackElement(card, self._driver)
            for card in cards
            if card.is_displayed() and card.text.strip()
        ]

    def _track_text_loaded(self, driver):
        return any(
            card.is_displayed() and card.text.strip()
            for card in driver.find_elements(*TrackListLocator.ITEM)
        )


### `TrackElement`: user-level actions

The useful abstraction is that higher-level code can say `.play()` and `.pause()` instead of knowing how the button is located.



In [ ]:
class TrackElement(WebComponent):
    def play(self):
        if not self.is_playing:
            self._get_play_button().click()

    def pause(self):
        if self.is_playing:
            self._get_play_button().click()

    @property
    def is_playing(self):
        return "Pause" in self._get_play_button().get_attribute(
            "aria-label"
        )

    def _get_play_button(self):
        return self._parent.find_element(
            *TrackLocator.PLAY_BUTTON
        )


The track object also extracts metadata. The important ideas are:

- retrieve the album URL from `href`
- strip a query string if present
- tolerate tracks without a genre
- return a clean representation of album/artist/genre/URL

The tutorial initially uses a dictionary and later replaces it with a `dataclass`.


In [ ]:
from selenium.common.exceptions import NoSuchElementException

def _get_track_info(self):
    full_url = self._parent.find_element(
        *TrackLocator.URL
    ).get_attribute("href")

    clean_url = full_url.split("?")[0] if full_url else ""

    try:
        genre = self._parent.find_element(
            *TrackLocator.GENRE
        ).text
    except NoSuchElementException:
        genre = ""

    return {
        "album": self._parent.find_element(*TrackLocator.ALBUM).text,
        "artist": self._parent.find_element(*TrackLocator.ARTIST).text,
        "genre": genre,
        "url": clean_url,
    }


### `locators.py`

This is a particularly useful organizational choice because selectors are likely to change independently of application logic.

Each locator is stored as a `(strategy, value)` tuple, so `*locator` can be passed directly into Selenium.


In [ ]:
from selenium.webdriver.common.by import By


class DiscoverPageLocator:
    DISCOVER_RESULTS = (By.CLASS_NAME, "results-grid")
    COOKIE_ACCEPT_NECESSARY = (
        By.CSS_SELECTOR,
        "#cookie-control-dialog button.g-button.outline",
    )


class TrackListLocator:
    ITEM = (By.CLASS_NAME, "results-grid-item")
    PAGINATION_BUTTON = (By.ID, "view-more")


class TrackLocator:
    PLAY_BUTTON = (
        By.CSS_SELECTOR,
        "button.play-pause-button",
    )
    URL = (By.CSS_SELECTOR, "div.meta p a")
    ALBUM = (By.CSS_SELECTOR, "div.meta p a strong")
    GENRE = (By.CSS_SELECTOR, "div.meta p.genre")
    ARTIST = (By.CSS_SELECTOR, "div.meta p a span")


### POM smoke test

Once the classes exist, the nice part is that the raw selectors disappear from the test/use site.


In [ ]:
from selenium.webdriver import Firefox
from bandcamp.pages import DiscoverPage

BANDCAMP_DISCOVER_URL = "https://bandcamp.com/discover/"

driver = Firefox()
driver.get(BANDCAMP_DISCOVER_URL)

page = DiscoverPage(driver)

track = page.discover_tracklist.available_tracks[0]
track.play()
track.pause()

driver.quit()


## 7. Build the Music Player App

At this point, the browser automation layer is mostly finished.

The tutorial then separates the application layer:

```text
bandcamp/
├── app/
│   ├── __init__.py
│   ├── player.py
│   └── tui.py
├── web/
│   ├── __init__.py
│   ├── base.py
│   ├── elements.py
│   ├── locators.py
│   └── pages.py
├── __init__.py
└── __main__.py
```

The important architectural split:

- `web/` = "How do I interact with Bandcamp?"
- `app/` = "What does my music player do?"


### Replace the metadata dictionary with a dataclass

A small `Track` dataclass makes the application's data easier to work with.


In [ ]:
from dataclasses import dataclass
from pprint import pformat


@dataclass
class Track:
    album: str
    artist: str
    genre: str
    url: str

    def __str__(self):
        return pformat(self)


Then `_get_track_info()` can return a `Track` instead of a dictionary:



In [ ]:
return Track(
    album=self._parent.find_element(*TrackLocator.ALBUM).text,
    artist=self._parent.find_element(*TrackLocator.ARTIST).text,
    genre=genre,
    url=clean_url,
)


### `player.py`

`Player` is now the high-level interface. It owns browser setup/teardown and delegates actual page interaction to the POM.

The context-manager methods are a nice touch because:

```python
with Player() as player:
    ...
```

automatically guarantees cleanup.


In [ ]:
from selenium.webdriver import Firefox
from selenium.webdriver.firefox.options import Options

from bandcamp.web.pages import DiscoverPage

BANDCAMP_DISCOVER_URL = "https://bandcamp.com/discover/"


class Player:
    def __init__(self):
        self._driver = self._set_up_driver()
        self.page = DiscoverPage(self._driver)
        self.tracklist = self.page.discover_tracklist
        self._current_track = self.tracklist.available_tracks[0]

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, exc_tb):
        self._driver.quit()

    def play(self, track_number=None):
        if track_number:
            self._current_track = (
                self.tracklist.available_tracks[track_number - 1]
            )
        self._current_track.play()

    def pause(self):
        self._current_track.pause()

    def _set_up_driver(self):
        options = Options()
        options.add_argument("--headless")

        browser = Firefox(options=options)
        browser.get(BANDCAMP_DISCOVER_URL)
        return browser


### `tui.py`: command loop

The tutorial uses Python 3.10+'s structural pattern matching.

This is a good example of where `match` is cleaner than a giant pile of `if/elif` statements.


In [ ]:
from bandcamp.app.player import Player

MAX_TRACKS = 100


def interact():
    with Player() as player:
        while True:
            print(
                "\nType: play [<track number>] | pause | "
                "tracks | more | exit"
            )

            match input("> ").strip().lower().split():
                case ["play"]:
                    play(player)

                case ["play", track]:
                    try:
                        play(player, int(track))
                    except ValueError:
                        print("Please provide a valid track number.")

                case ["pause"]:
                    pause(player)

                case ["tracks"]:
                    display_tracks(player)

                case ["more"] if (
                    len(player.tracklist.available_tracks) >= MAX_TRACKS
                ):
                    print("Can't load more tracks.")

                case ["more"]:
                    player.tracklist.load_more()
                    display_tracks(player)

                case ["exit"]:
                    print("Exiting the player...")
                    break

                case _:
                    print("Unknown command. Try again.")


### TUI helper functions

The `play()` helper catches invalid indices, while `display_tracks()` turns the track objects into a simple terminal table.


In [ ]:
def play(player, track_number=None):
    try:
        player.play(track_number)
        print(player._current_track._get_track_info())
    except IndexError:
        print(
            "Please provide a valid track number. "
            "You can list tracks with `tracks`."
        )


def pause(player):
    player.pause()


COLUMN_WIDTH = 30
CW = COLUMN_WIDTH


def display_tracks(player):
    header = (
        f"{'#':<5} {'Album':<{CW}} "
        f"{'Artist':<{CW}} {'Genre':<{CW}}"
    )
    print(header)
    print("-" * 80)

    for number, element in enumerate(
        player.tracklist.available_tracks,
        start=1,
    ):
        track = element._get_track_info()

        album = _truncate(track.album, CW)
        artist = _truncate(track.artist, CW)
        genre = _truncate(track.genre, CW)

        print(
            f"{number:<5} {album:<{CW}} "
            f"{artist:<{CW}} {genre:<{CW}}"
        )


def _truncate(text, width):
    return text[: width - 3] + "..." if len(text) > width else text


### `__main__.py`

The package gets a tiny entry point:


In [ ]:
from bandcamp.app.tui import interact


def main():
    interact()


### `pyproject.toml`

The tutorial uses a package entry point so the app can eventually be installed and invoked with a simple command.


In [ ]:
[build-system]
requires = ["setuptools", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "bandcamp_player"
version = "0.1.0"
requires-python = ">=3.10"
description = "A web player for Bandcamp using Selenium"
dependencies = [
    "selenium",
]

[project.scripts]
discover = "bandcamp.__main__:main"


Then install the project locally:



In [ ]:
# From the folder containing pyproject.toml:
python -m pip install .

# Then:
discover


## 8. What I Actually Want to Remember

### Selenium fundamentals

```text
WebDriver
  ↓
get(url)
  ↓
find_element / find_elements
  ↓
WebElement
  ↓
click / send_keys / submit / get_attribute / text
```

### Waiting

**Implicit wait:** broad safety net.

**Explicit wait:** wait for a specific condition.

**`time.sleep()`:** okay for quick demonstrations/debugging, but usually the least robust option.

### POM

```text
Application
    ↓
Page object
    ↓
Component object
    ↓
Locator
    ↓
Selenium WebDriver
    ↓
Actual browser
```

The application should not need to know that the play button happens to be a CSS selector today.

### Main lessons from the project

1. Inspect the DOM before inventing selectors.
2. Prefer stable locators.
3. Expect dynamic content.
4. Wait for conditions, not arbitrary durations.
5. Handle optional UI such as cookie banners defensively.
6. Put brittle selectors in one place.
7. Wrap repeated browser behavior in page/component objects.
8. Keep application/business logic separate from browser mechanics.
9. Always clean up WebDriver instances.
10. Debug headful before blaming headless mode.

### One particularly useful mental distinction

**Selenium code answers:**  
> "How do I make the website do this?"

**Application code answers:**  
> "What does my program want to do?"

POM is the boundary between those two questions.


## 9. Possible Extensions

The tutorial suggests several directions for extending the player, including:

- searching/filtering by genre
- improving the text UI
- adding more robust input validation
- writing tests against the page objects
- handling more sophisticated browser behavior

My own takeaway is that the interesting part isn't really the music player. It's the pattern for turning a messy, changing website into a reasonably clean Python interface.

**Source:** Martin Breuss, "Modern Web Automation With Python and Selenium," Real Python, Apr. 30, 2025.
